# Fault section buffers

Builds hanging-wall, footwall, and/or along-strike edge buffers around fault
sections, joining subsections of the same named fault into one continuous,
gap-free shape (using shared vertices between adjacent rectangle patches --
no morphological gap-closing hacks needed).

- **trace_or_plane = 'trace'**: buffers just the fault surface trace (its
  top/shallow edge).
- **trace_or_plane = 'plane'**: buffers the whole fault-plane footprint
  (its full up-dip-to-down-dip extent).

Any of `footbuff`, `hangingbuff`, `edgebuff` can be set to 0 to switch that
buffer off (e.g. edge buffers only, or hanging-wall only).

Hanging-wall/footwall/outward sides are worked out directly from each
rectangle's own geometry (comparing its shallow edge to its deep edge) --
not from the `DipDir` attribute -- so it isn't sensitive to any errors in
that metadata.

Outputs (buffered geometry, un-buffered base geometry, and a run-metadata
JSON) are written to `outputdir`.


In [ ]:
import os
import re
import json
import datetime as dt

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon, LineString, Point
from shapely.ops import unary_union
import matplotlib.pyplot as plt

os.getcwd()


In [ ]:
# ---------------------------------------------------------------------------
# Parameters
# ---------------------------------------------------------------------------
fault_type = 'crustal'      # 'crustal' / 'CFM', 'py' / 'puysegur', or 'sz' (Hikurangi-Kermadec)
fault_name = None           # None for all faults, or e.g. 'Wairarapa: 3'. Ignored for sz/py
                             # (they have no fault names -- see note below).
trace_or_plane = 'plane'    # 'trace' -> buffer just the surface trace
                             # 'plane' -> buffer the whole fault-plane footprint

# Only used for fault_type='crustal' and trace_or_plane='trace'. 'mesh' (the
# default) reads the trace line from the collapsed-mesh file
# (crustal_discretised_polygons.geojson); 'rectangle' reads it from the
# rectangle patches' own top edge instead. The two give slightly different
# trace shapes, but the hanging-wall side is always resolved from the
# rectangles either way, so switching this doesn't change which side is
# which.
trace_source = 'mesh'

# Buffer distances in metres. Set any of these to 0 to switch that buffer off.
hangingbuff = 3000    # hanging-wall-side buffer
footbuff = 0       # footwall-side buffer (ignored -> uses hangingbuff for vertical (90 deg) faults)
edgebuff = 1500      # along-strike buffer at the two true ends (tips) of each named fault.
                        # Not supported for sz/py (see note below) -- always 0 there.

outputdir = './sites/poly_buffers'

# Note on sz / py: those files (e.g. sz_all_rectangle_outlines.geojson) are a
# flat set of rectangle patches tiling the whole subduction interface, with
# no fault_name/subsection numbering and no Z coordinate -- unlike the
# crustal file, there's no way to group them into named faults or chain them
# in along-strike order. So for sz/py: every patch is buffered individually
# (footwall/hanging-wall only, from its own geometry) and unioned into one
# combined interface buffer; `fault_name`, `trace_source`, and `edgebuff` are
# all ignored in that case.


In [ ]:
# ---------------------------------------------------------------------------
# Core geometry functions: build gap-free, joined hanging-wall / footwall /
# edge buffers around fault sections, either around just the trace or the
# whole fault-plane footprint.
#
# Two paths:
#  - build_fault_buffers(): named, subsection-chained faults (crustal / CFM),
#    where subsections share exact corner vertices and can be ordered by name.
#  - build_gridded_buffers(): sz / py subduction-interface rectangle files,
#    which have no fault_name/subsection numbering or Z coordinate -- every
#    patch is buffered individually and unioned, which needs no ordering.
# ---------------------------------------------------------------------------



SUBSECTION_RE = re.compile(r",\s*Subsection\s*(\d+)\s*$")

# Which of the 4 ring vertices [c0, c1, c2, c3] (from a closed 5-point
# rectangle ring [c0, c1, c2, c3, c0]) correspond to (top1, top2, bottom1,
# bottom2), per fault type. Crustal (CFM) rectangles are ordered
# [top1, top2, bottom1, bottom2]. Subduction interface rectangles use a
# different winding, so are relabelled here rather than by physically
# reordering coordinates.
CORNER_ORDER = {
    "CFM": (0, 1, 2, 3),
    "crustal": (0, 1, 2, 3),
    "sz": (1, 2, 3, 0),
    "py": (0, 1, 2, 3),
}

CHAIN_TOL_M = 1.0  # tolerance for treating two subsection corners as "the same point"

# Parameters for extracting a trace line out of the collapsed-mesh file
# (crustal_discretised_polygons.geojson) instead of a rectangle's top edge.
SURFACE_Z_TOL_M = 50.0            # vertices within this of z=0 count as "on the trace"
BLIND_FAULT_TOP_FRACTION = 0.0   # fallback fraction for blind faults with no z=0 vertex
MESH_CHAIN_TOL_M = 1000.0         # mesh subsections aren't guaranteed to share exact
                                   # vertices like the rectangles do, so this is looser
                                   # than CHAIN_TOL_M -- it's just used to flag a real gap


def strip_subsection(fault_name):
    return SUBSECTION_RE.sub("", fault_name).strip()


def subsection_number(fault_name):
    m = SUBSECTION_RE.search(fault_name)
    return int(m.group(1)) if m else 0


def get_rectangle_corners(polygon, corner_order):
    """Return (top1, top2, bottom1, bottom2) as 2D numpy arrays for one fault-patch rectangle."""
    coords = np.array(polygon.exterior.coords)[:4, :2]  # drop the closing repeat + Z
    i0, i1, i2, i3 = corner_order
    return coords[i0], coords[i1], coords[i2], coords[i3]


# ---------------------------------------------------------------------------
# Mesh-sourced trace extraction (crustal_discretised_polygons.geojson) --
# used by default for crustal 'trace' mode instead of the rectangles' top edge.
# ---------------------------------------------------------------------------
def get_polygon_parts(geometry):
    if geometry.geom_type == "Polygon":
        return [geometry]
    elif geometry.geom_type == "MultiPolygon":
        return list(geometry.geoms)
    return []


def _order_points_along_principal_axis(points):
    if len(points) < 2:
        return points
    centroid = points.mean(axis=0)
    centered = points - centroid
    cov = np.cov(centered.T)
    if np.isnan(cov).any():
        return points
    eigvals, eigvecs = np.linalg.eigh(cov)
    principal = eigvecs[:, np.argmax(eigvals)]
    order = np.argsort(centered @ principal)
    return points[order]


def trace_from_mesh_part(polygon):
    """Extract an ordered (x, y) trace line from one collapsed-mesh polygon part."""
    coords = np.array(polygon.exterior.coords)
    if coords.shape[1] < 3:
        return None
    xy, z = coords[:, :2], coords[:, 2]

    on_surface = np.abs(z) <= SURFACE_Z_TOL_M
    if on_surface.sum() >= 2:
        trace_mask = on_surface
    else:
        return None
        z_max, z_min = z.max(), z.min()
        cutoff = z_max - BLIND_FAULT_TOP_FRACTION * (z_max - z_min)
        trace_mask = z >= cutoff
        if trace_mask.sum() < 2:
            trace_mask = z >= np.sort(z)[::-1][:2].min()

    trace_pts = xy[trace_mask]
    if len(trace_pts) < 2:
        return None
    ordered = _order_points_along_principal_axis(trace_pts)

    pts = []
    for pt in ordered:
        if not pts or not np.allclose(pt, pts[-1], atol=1e-6):
            pts.append(pt)
    return pts if len(pts) >= 2 else None


def trace_from_mesh_geometry(geometry):
    """Extract a trace line from a mesh feature's geometry (Polygon or MultiPolygon)."""
    for part in get_polygon_parts(geometry):
        pts = trace_from_mesh_part(part)
        if pts is not None:
            return pts
    return None


def build_mesh_trace_chain(ordered_mesh_geoms):
    """
    Chain the per-subsection mesh trace lines (already in along-strike order)
    into one continuous polyline. Each subsection's own trace points aren't
    guaranteed to be oriented consistently or to share exact vertices with
    its neighbours (unlike the rectangles), so consecutive segments are
    oriented to connect end-to-start and just joined directly.

    Returns (chain, is_continuous).
    """
    chain = []
    is_continuous = True

    for geom in ordered_mesh_geoms:
        pts = trace_from_mesh_geometry(geom)
        if pts is None:
            is_continuous = False
            continue
        if not chain:
            chain.extend(pts)
            continue
        d_start = np.linalg.norm(np.array(pts[0]) - np.array(chain[-1]))
        d_end = np.linalg.norm(np.array(pts[-1]) - np.array(chain[-1]))
        if d_end < d_start:
            pts = pts[::-1]
            d_start = d_end
        if d_start > MESH_CHAIN_TOL_M:
            is_continuous = False
        chain.extend(pts)

    return chain, is_continuous


def build_chains(ordered_polygons, corner_order):
    """
    Build continuous top-edge and bottom-edge chains (as lists of 2D points)
    for a sequence of subsection rectangles, ordered along strike.

    Returns (top_chain, bottom_chain, is_continuous). is_continuous is False
    if any pair of adjacent subsections doesn't share a corner within
    CHAIN_TOL_M (i.e. there's a genuine gap between subsections in the
    source data).
    """
    top_chain = []
    bottom_chain = []
    is_continuous = True

    for i, polygon in enumerate(ordered_polygons):
        top1, top2, bottom1, bottom2 = get_rectangle_corners(polygon, corner_order)
        if i == 0:
            top_chain.append(top1)
            bottom_chain.append(bottom2)
        else:
            if np.linalg.norm(top1 - top_chain[-1]) > CHAIN_TOL_M:
                is_continuous = False
            if np.linalg.norm(bottom2 - bottom_chain[-1]) > CHAIN_TOL_M:
                is_continuous = False
        top_chain.append(top2)
        bottom_chain.append(bottom1)

    return top_chain, bottom_chain, is_continuous


def single_sided_sign(chain, inward_vector):
    """
    Sign (+1/-1) to pass to shapely's `single_sided=True` buffer of the
    LineString `chain` (walked from chain[0] to chain[-1]) so that a
    *positive* buffer distance times this sign lands on the side AWAY FROM
    `inward_vector`'s direction (i.e. "outward").
    """
    edge_dir = np.array(chain[1]) - np.array(chain[0])
    edge_dir = edge_dir / np.linalg.norm(edge_dir)
    left_normal = np.array([-edge_dir[1], edge_dir[0]])
    return -1.0 if np.dot(inward_vector, left_normal) > 0 else 1.0


def buffer_chain(chain, distance, sign):
    """Single-sided buffer of a chain. distance<=0 -> no buffer (returns None)."""
    if distance <= 0:
        return None
    line = LineString([tuple(p) for p in chain])
    return line.buffer(distance * sign, single_sided=True)


def build_fault_geometry(name, ordered_polygons, corner_order,
                          trace_or_plane, footbuff, hangingbuff, edgebuff,
                          dip_deg=None, trace_chain=None):
    """
    Build the buffered geometry for one named fault from its ordered list of
    subsection rectangle polygons (shapely Polygons with Z, exterior 5-pt ring).

    trace_chain : optional pre-built chain of 2D points to use as the trace
        line instead of the rectangles' own top edge (used when the trace
        is sourced from the collapsed-mesh file instead of the rectangles).
        Ignored in 'plane' mode (the plane always comes from the rectangles).

    Returns (buffered_geometry, base_geometry, is_continuous)
    """
    top_chain, bottom_chain, is_continuous = build_chains(ordered_polygons, corner_order)

    # Reference "inward" direction (from top chain towards bottom chain), used to
    # work out which side of each chain is outward (footwall / hanging wall).
    # This always comes from the rectangles (even when the trace itself is
    # sourced from the mesh) since it's the rectangles that encode dip.
    top_centroid = np.mean(top_chain, axis=0)
    bottom_centroid = np.mean(bottom_chain, axis=0)
    inward_vector = bottom_centroid - top_centroid  # points from top chain towards bottom chain

    footwall_sign = single_sided_sign(top_chain, inward_vector)

    # A vertical fault (dip 90) has no real hanging-wall/footwall distinction;
    # fall back to the footwall distance on both sides in that case.
    effective_footbuff = footbuff
    if dip_deg is not None and np.isclose(dip_deg, 90.0):
        effective_footbuff = hangingbuff

    pieces = []

    if trace_or_plane == "trace":
        # Both buffers apply to the SAME trace line (one on each side) --
        # NOT to the far-away deep edge, which for shallow-dipping faults
        # can be many km away in map view.
        line_chain = trace_chain if trace_chain is not None else top_chain
        line_sign = single_sided_sign(line_chain, inward_vector) if trace_chain is not None else footwall_sign

        footwall_buf = buffer_chain(line_chain, effective_footbuff, line_sign)
        if footwall_buf is not None:
            pieces.append(footwall_buf)
        hangingwall_buf = buffer_chain(line_chain, hangingbuff, -line_sign)
        if hangingwall_buf is not None:
            pieces.append(hangingwall_buf)

        base_polygon = LineString(line_chain)
        tip_points = [line_chain[0], line_chain[-1]]
    else:
        footwall_buf = buffer_chain(top_chain, effective_footbuff, footwall_sign)
        if footwall_buf is not None:
            pieces.append(footwall_buf)

        # hanging wall is buffered from the bottom chain, walked in the SAME
        # along-strike order as the top chain (see build_chains), so its
        # inward reference is reversed relative to the top chain's.
        hangingwall_sign = single_sided_sign(bottom_chain, -inward_vector)
        hangingwall_buf = buffer_chain(bottom_chain, hangingbuff, hangingwall_sign)
        if hangingwall_buf is not None:
            pieces.append(hangingwall_buf)

        base_polygon = Polygon(list(top_chain) + list(reversed(bottom_chain)))
        pieces.append(base_polygon)
        tip_points = None  # plane mode uses the perpendicular tip edges below instead

    # Edge buffers: a symmetric buffer of radius `edgebuff` around the true
    # along-strike ends of the whole fault. This extends the fault lengthwise
    # at its tips, independent of whether footwall/hanging-wall buffers are
    # switched on. In 'plane' mode this is a buffer around the perpendicular
    # (top-to-bottom) edge at each tip; in 'trace' mode there's no bottom
    # edge to pair with, so it's just a buffer around the tip point itself.
    if edgebuff > 0:
        if tip_points is not None:
            pieces.append(Point(tip_points[0]).buffer(edgebuff))
            pieces.append(Point(tip_points[-1]).buffer(edgebuff))
        else:
            start_edge = LineString([tuple(top_chain[0]), tuple(bottom_chain[0])])
            end_edge = LineString([tuple(top_chain[-1]), tuple(bottom_chain[-1])])
            pieces.append(start_edge.buffer(edgebuff))
            pieces.append(end_edge.buffer(edgebuff))

    if not pieces:
        return None, None, is_continuous

    buffered = unary_union(pieces)
    if trace_chain is not None:
        # Mesh-sourced traces aren't guaranteed to be a perfectly smooth
        # polyline (adjacent subsections' independently-derived trace
        # segments can kink slightly), which can leave a thin self-
        # intersecting sliver in the buffered result. A small morphological
        # closing (dilate then erode by an amount much smaller than the
        # buffer widths themselves) cleans that up without changing the
        # true buffer extent.
        MESH_CLEANUP_GAP_M = 200.0
        buffered = buffered.buffer(MESH_CLEANUP_GAP_M).buffer(-MESH_CLEANUP_GAP_M)
    return buffered, base_polygon, is_continuous


def load_polygons(polygon_path):
    gdf = gpd.read_file(polygon_path)
    gdf["fault_id"] = gdf["fault_id"].astype(int)
    return gdf.set_index("fault_id")


def load_mesh_polygons(mesh_path):
    """
    Like load_polygons, but the collapsed-mesh file can have more than one
    row per fault_id (a handful of multi-part fault patches) -- those get
    unioned into a single (Multi)Polygon per fault_id first.
    """
    gdf = gpd.read_file(mesh_path)
    gdf["fault_id"] = gdf["fault_id"].astype(int)
    if gdf["fault_id"].duplicated().any():
        gdf = gdf.dissolve(by="fault_id", as_index=False)
    return gdf.set_index("fault_id")


# ---------------------------------------------------------------------------
# sz / py subduction-interface rectangle files: no fault_name/subsection
# numbering and no Z coordinate, so subsections can't be grouped into named
# faults or chained in along-strike order the way the crustal file can be.
# Instead, every patch is buffered individually (its own top/bottom edges,
# outward-from-centroid, using CORNER_ORDER) and unioned together -- any
# buffer that lands on a shared/internal edge between two patches just gets
# swallowed into the base polygon, so this naturally produces one seamless,
# gap-free result across the whole interface without needing to know patch
# adjacency or ordering.
#
# There's no reliable concept of "the along-strike ends of the fault" for a
# full 2D interface mesh like this (its true edge is the whole perimeter,
# not two tips), so edge buffers aren't supported in this path.
# ---------------------------------------------------------------------------
def build_gridded_buffers(polygon_gdf, corner_order,
                           trace_or_plane="plane",
                           footbuff=3000.0, hangingbuff=5000.0, edgebuff=3000.0,
                           group_name="interface"):
    footwall_pieces = []
    hangingwall_pieces = []
    edge_pieces = []
    base_pieces = []

    for fault_id, row in polygon_gdf.iterrows():
        polygon = row.geometry
        top1, top2, bottom1, bottom2 = get_rectangle_corners(polygon, corner_order)
        centroid = np.mean([top1, top2, bottom1, bottom2], axis=0)

        dip_deg = row.get("dip_deg")
        effective_footbuff = footbuff
        if dip_deg is not None and np.isclose(dip_deg, 90.0):
            effective_footbuff = hangingbuff

        top_chain = [top1, top2]
        bottom_chain = [bottom1, bottom2]  # NB: walked bottom1->bottom2 here (per-patch, no whole-fault ordering)
        edge_chain1 = [top1, bottom2]
        edge_chain2 = [top2, bottom1]

        footwall_sign = single_sided_sign(top_chain, centroid - np.mean(top_chain, axis=0))
        hangingwall_sign = single_sided_sign(bottom_chain, centroid - np.mean(bottom_chain, axis=0))
        edge1_sign = single_sided_sign(edge_chain1, centroid - np.mean(edge_chain1, axis=0))
        edge2_sign = single_sided_sign(edge_chain2, centroid - np.mean(edge_chain2, axis=0))

        if effective_footbuff > 0:
            footwall_pieces.append(buffer_chain(top_chain, effective_footbuff, footwall_sign))
        if hangingbuff > 0:
            hangingwall_pieces.append(buffer_chain(bottom_chain, hangingbuff, hangingwall_sign))
        if edgebuff > 0:
            edge_pieces.append(buffer_chain(edge_chain1, edgebuff, edge1_sign))
            edge_pieces.append(buffer_chain(edge_chain2, edgebuff, edge2_sign))
        if trace_or_plane == "plane":
            base_pieces.append(polygon)

    pieces = [p for p in footwall_pieces + hangingwall_pieces + edge_pieces if p is not None]
    if trace_or_plane == "plane":
        base_polygon = unary_union(base_pieces)
        pieces.append(base_polygon)
    else:
        base_polygon = unary_union([LineString(get_rectangle_corners(g, corner_order)[:2])
                                     for g in polygon_gdf.geometry])

    buffered = unary_union(pieces) if pieces else None

    buff_gdf = gpd.GeoDataFrame(
        [{"FaultName": group_name, "n_subsections": len(polygon_gdf), "trace_or_plane": trace_or_plane,
          "footwall_buffer_m": footbuff, "hangingwall_buffer_m": hangingbuff, "edge_buffer_m": 0}],
        geometry=[buffered], crs=polygon_gdf.crs,
    )
    base_gdf = gpd.GeoDataFrame(
        [{"FaultName": group_name}], geometry=[base_polygon], crs=polygon_gdf.crs,
    )
    return buff_gdf, base_gdf


def build_fault_buffers(polygon_gdf, traces_df,
                         trace_or_plane="plane",
                         footbuff=3000.0, hangingbuff=5000.0, edgebuff=2000.0,
                         fault_type="CFM", fault_name=None,
                         trace_source="mesh", mesh_polygon_gdf=None):
    """
    polygon_gdf : GeoDataFrame indexed by fault_id, one rectangle per subsection
    traces_df   : DataFrame/GeoDataFrame with columns FaultID, ParentName,
                  DipDeg (optional), and anything else you want carried
                  through as metadata (e.g. SlipRate, DipDir)
    trace_source : "mesh" (default) or "rectangle". Only relevant when
                  trace_or_plane == "trace". "mesh" reads the trace line from
                  the collapsed-mesh file (mesh_polygon_gdf, indexed by
                  fault_id) instead of the rectangles' own top edge -- the
                  two sources' traces differ slightly, but the hanging-wall
                  side is always resolved from the rectangles regardless, so
                  switching source doesn't change which side is which.
    mesh_polygon_gdf : GeoDataFrame indexed by fault_id, required when
                  trace_source == "mesh" and trace_or_plane == "trace".
    """
    corner_order = CORNER_ORDER.get(fault_type, (0, 1, 2, 3))
    use_mesh = trace_or_plane == "trace" and trace_source == "mesh"
    if use_mesh and mesh_polygon_gdf is None:
        raise ValueError("trace_source='mesh' requires mesh_polygon_gdf (the collapsed-mesh file)")

    if fault_name:
        traces_df = traces_df[traces_df["ParentName"] == fault_name]

    out_rows = []
    out_geoms = []
    out_base_geoms = []
    warnings = []

    for name, group in traces_df.groupby("ParentName"):
        group = group.copy()
        group["_subsection_n"] = group["ParentName"].index.map(lambda i: 0)  # placeholder, overwritten below
        # order subsections along strike using the parsed subsection number
        # from FaultName if available, else FaultID order
        if "FaultName" in group.columns:
            group["_subsection_n"] = group["FaultName"].map(subsection_number)
        group = group.sort_values("_subsection_n")

        ordered_polygons = [polygon_gdf.loc[int(fid)].geometry for fid in group["FaultID"]]
        dip_deg = group["DipDeg"].iloc[0] if "DipDeg" in group.columns else None

        trace_chain, mesh_continuous = (None, True)
        if use_mesh:
            ordered_mesh_geoms = [mesh_polygon_gdf.loc[int(fid)].geometry for fid in group["FaultID"]
                                   if int(fid) in mesh_polygon_gdf.index]
            if len(ordered_mesh_geoms) == len(group):
                trace_chain, mesh_continuous = build_mesh_trace_chain(ordered_mesh_geoms)
                if len(trace_chain) < 2:
                    trace_chain = None
            else:
                mesh_continuous = False  # missing mesh rows for this fault -> fall back below

        geom, base_geom, is_continuous = build_fault_geometry(
            name, ordered_polygons, corner_order,
            trace_or_plane, footbuff, hangingbuff, edgebuff, dip_deg=dip_deg,
            trace_chain=trace_chain,
        )
        is_continuous = is_continuous and mesh_continuous
        if geom is None:
            continue
        if not is_continuous:
            warnings.append(name)

        out_geoms.append(geom)
        out_base_geoms.append(base_geom)
        row = {
            "FaultName": name,
            "n_subsections": len(group),
            "trace_or_plane": trace_or_plane,
            "trace_source": trace_source if trace_or_plane == "trace" else "rectangle",
            "footwall_buffer_m": footbuff,
            "hangingwall_buffer_m": hangingbuff,
            "edge_buffer_m": edgebuff,
            "is_continuous": is_continuous,
        }
        for col in ("DipDir", "DipDeg", "SlipRate"):
            if col in group.columns:
                row[col] = group[col].iloc[0]
        out_rows.append(row)

    buff_gdf = gpd.GeoDataFrame(out_rows, geometry=out_geoms, crs=polygon_gdf.crs)
    base_gdf = gpd.GeoDataFrame(out_rows, geometry=out_base_geoms, crs=polygon_gdf.crs)

    if warnings:
        tol_note = f"{MESH_CHAIN_TOL_M} m (mesh trace)" if use_mesh else f"{CHAIN_TOL_M} m"
        print(f"Warning: {len(warnings)} fault(s) had non-contiguous subsections "
              f"(gap larger than {tol_note} between adjacent patches): {warnings[:5]}"
              + (" ..." if len(warnings) > 5 else ""))

    return buff_gdf, base_gdf


In [ ]:
# ---------------------------------------------------------------------------
# Locate input files and load them
# ---------------------------------------------------------------------------
is_gridded = fault_type.lower() in ('py', 'puysegur', 'sz')

if fault_type.lower() in ('crustal', 'cfm'):
    fault_type = 'CFM'
    datadir = "./crustal/discretised_CFM"
    polygon_file = "named_rectangle_polygons.geojson"
    trace_file = "name_filtered_fault_sections.geojson"
    mesh_file = "crustal_discretised_polygons.geojson"
elif fault_type.lower() in ('py', 'puysegur'):
    fault_type = 'py'
    datadir = "./subduction/discretised_puysegur"
    polygon_file = "py_all_rectangle_outlines.geojson"
else:
    fault_type = 'sz'
    datadir = "./subduction/discretised_fq_hikkerm"
    polygon_file = "sz_all_rectangle_outlines.geojson"

polygon_path = f"{datadir}/{polygon_file}"

if is_gridded:
    # sz / py: flat rectangle-patch files only -- fault_id, dip_deg,
    # up_depth_km, low_depth_km, geometry. No fault_name, no Z.
    polygon_gdf = load_polygons(polygon_path)
    outname = f"{fault_type}-{trace_or_plane}_hang-{hangingbuff}m_foot-{footbuff}m_edge-{edgebuff}m"
    print(f"fault_type={fault_type} (gridded interface), trace_or_plane={trace_or_plane}, outname={outname}")
    print(f"{len(polygon_gdf)} interface patches, no fault-name grouping available "
          f"-> buffered as one combined interface")

else:
    trace_path = f"{datadir}/{trace_file}"
    mesh_path = f"{datadir}/{mesh_file}"

    polygon_gdf = load_polygons(polygon_path)
    mesh_polygon_gdf = load_mesh_polygons(mesh_path) if os.path.exists(mesh_path) else None
    if trace_or_plane == 'trace' and trace_source == 'mesh' and mesh_polygon_gdf is None:
        print(f"trace_source='mesh' but {mesh_path} wasn't found -- falling back to trace_source='rectangle'")
        trace_source = 'rectangle'

    if os.path.exists(trace_path):
        # Real fault-sections file, with metadata like SlipRate that isn't in
        # the rectangle polygons themselves.
        traces_df = gpd.read_file(trace_path).to_crs(2193)
        rename_map = {}
        for src, dst in [("fault_id", "FaultID"), ("dip_dir_deg", "DipDir"), ("dip_deg", "DipDeg")]:
            if src in traces_df.columns and dst not in traces_df.columns:
                rename_map[src] = dst
        traces_df = traces_df.rename(columns=rename_map)
        if "ParentName" not in traces_df.columns and "fault_name" in traces_df.columns:
            traces_df["ParentName"] = traces_df["fault_name"].map(lambda n: re.sub(r",\s*Subsection\s*\d+$", "", n).strip())
        if "FaultName" not in traces_df.columns and "fault_name" in traces_df.columns:
            traces_df["FaultName"] = traces_df["fault_name"]
    else:
        # No separate trace/section file available -- the rectangle polygon
        # file already carries fault_id, fault_name (with ", Subsection N"),
        # dip_deg and dip_dir_deg, which is everything the buffering logic
        # needs (just no SlipRate or other extra section metadata).
        print(f"No trace/section file found at {trace_path} -- deriving fault-section "
              f"metadata directly from {polygon_path} instead.")
        raw = gpd.read_file(polygon_path).to_crs(2193)
        traces_df = pd.DataFrame({
            "FaultID": raw["fault_id"].astype(int),
            "FaultName": raw["fault_name"],
            "ParentName": raw["fault_name"].map(lambda n: re.sub(r",\s*Subsection\s*\d+$", "", n).strip()),
            "DipDir": raw["dip_dir_deg"],
            "DipDeg": raw["dip_deg"],
        })

    if fault_name:
        outname = f"{fault_name.replace(' ', '-')}-{trace_or_plane}_hang-{hangingbuff}m_foot-{footbuff}m_edge-{edgebuff}m"
    else:
        outname = f"{fault_type}-{trace_or_plane}_hang-{hangingbuff}m_foot-{footbuff}m_edge-{edgebuff}m"
    if trace_or_plane == 'trace':
        outname += f"_{trace_source}"

    print(f"fault_type={fault_type}, trace_or_plane={trace_or_plane}, outname={outname}")
    print(f"{len(polygon_gdf)} fault-patch rectangles, {traces_df['ParentName'].nunique()} named faults")


In [ ]:
# ---------------------------------------------------------------------------
# Build the buffers and write outputs (geometry + metadata) to `outputdir`
# ---------------------------------------------------------------------------
os.makedirs(outputdir, exist_ok=True)

if is_gridded:
    buff_gdf, base_gdf = build_gridded_buffers(
        polygon_gdf, CORNER_ORDER.get(fault_type, (0, 1, 2, 3)),
        trace_or_plane=trace_or_plane,
        footbuff=footbuff, hangingbuff=hangingbuff, edgebuff=edgebuff,
        group_name=fault_type,
    )
else:
    buff_gdf, base_gdf = build_fault_buffers(
        polygon_gdf, traces_df,
        trace_or_plane=trace_or_plane,
        footbuff=footbuff, hangingbuff=hangingbuff, edgebuff=edgebuff,
        fault_type=fault_type, fault_name=fault_name,
        trace_source=trace_source, mesh_polygon_gdf=mesh_polygon_gdf,
    )

buffer_path = f"{outputdir}/{outname}.geojson"
base_path = f"{outputdir}/{fault_type}_faultsections_{trace_or_plane}.geojson"
metadata_path = f"{outputdir}/{outname}_metadata.json"

buff_gdf.to_file(buffer_path, driver="GeoJSON")
base_gdf.to_file(base_path, driver="GeoJSON")

run_metadata = {
    "created": dt.datetime.now().isoformat(timespec="seconds"),
    "fault_type": fault_type,
    "is_gridded_interface": is_gridded,
    "fault_name_filter": None if is_gridded else fault_name,
    "trace_or_plane": trace_or_plane,
    "trace_source": (trace_source if (trace_or_plane == "trace" and not is_gridded) else None),
    "footwall_buffer_m": footbuff,
    "hangingwall_buffer_m": hangingbuff,
    "edge_buffer_m": (0 if is_gridded else edgebuff),
    "polygon_source": polygon_path,
    "n_faults": int(len(buff_gdf)),
    "buffer_output": buffer_path,
    "base_geometry_output": base_path,
}
if not is_gridded:
    run_metadata["n_subsections_total"] = int(buff_gdf["n_subsections"].sum())
    run_metadata["faults_with_gaps_between_subsections"] = buff_gdf.loc[~buff_gdf["is_continuous"], "FaultName"].tolist()

with open(metadata_path, "w") as f:
    json.dump(run_metadata, f, indent=2)

print(f"Wrote {len(buff_gdf)} buffered fault(s) -> {buffer_path}")
print(f"Wrote base (un-buffered) fault geometry -> {base_path}")
print(f"Wrote run metadata -> {metadata_path}")
if not is_gridded and run_metadata["faults_with_gaps_between_subsections"]:
    print(f"NOTE: {len(run_metadata['faults_with_gaps_between_subsections'])} fault(s) had a gap between "
          f"subsections bigger than the join tolerance (mesh traces are looser than rectangle corners, "
          f"so this is expected more often with trace_source='mesh'; the trace is still joined by a "
          f"straight connector across the gap): {run_metadata['faults_with_gaps_between_subsections'][:10]}")


In [ ]:
# ---------------------------------------------------------------------------
# Quick visual check
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 11))
buff_gdf.plot(ax=ax, color="orange", alpha=0.5)
base_gdf.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=0.4)
ax.set_aspect("equal")
ax.set_title(outname)
plt.show()
